# Lecture 21: the live data-change demo


| Act | What this notebook writes | What we expect |
| --- | --- | --- |
| 1 | `heat_spike_wilmington.json`: a real Wilmington file with 45 °C | A `HEAT` row in `weather_alerts`; map turns hot; `check_heat_alert` triggers |
| 2 | `bad_city.json` (no city) | Row dropped by `valid_city`; the job succeeds; nothing changes in gold |


# 0. Pause the scheduling of the job
pause the hourly schedule of `weather-hourly`, so no real data lands in the middle of the demo. Attach serverless compute.

# 1. Settings

The files go in the **next** hour's folder. The current hour already has a real Wilmington reading, and `gold_hourly_city` would average 45 °C with it.

In [0]:
import json, os, glob
from datetime import datetime, timezone, timedelta

LANDING = "/Volumes/dsc592/weather/landing/weather"
CITY    = "Wilmington"                      # one of the cities in YOUR cities.csv
slug    = CITY.lower().replace(" ", "_")    # the file name ingest uses: "New York" -> new_york

# The latest real file of that city: we copy it, so the layout matches exactly
# sorted() sorts alphabetically
latest = sorted(glob.glob(f"{LANDING}/*/*/*/*/{slug}.json"))[-1]
print("Copying:", latest)

now    = datetime.now(timezone.utc)
hour   = now.replace(minute=0, second=0, microsecond=0) + timedelta(hours=1)   # the NEXT hour
folder = f"{LANDING}/{hour:%Y/%m/%d/%H}"
print("Writing to:", folder)

# 2. Act 1: The heat spike


In [0]:
rec = json.load(open(latest))  # copy the latest file to rec
rec["fetched_at"] = now.isoformat(timespec="seconds")  # change fetched_at to now

# hour.strftime("%Y-%m-%dT%H:%M") gives the NEXT hour
rec["payload"]["current"]["time"] = hour.strftime("%Y-%m-%dT%H:%M")

# fake heat spike
rec["payload"]["current"]["temperature_2m"] = 45.0

os.makedirs(folder, exist_ok=True)
with open(f"{folder}/heat_spike_{slug}.json", "w") as f: 
    json.dump(rec, f)

print(os.listdir(folder))

## 2.1. Now: run the ETL pipeline or the job `weather-hourly`. 

How the data flow:
```text
            WHEN (weather)          HOW HOT                  WHEN (loaded)
LANDING     "time":                 "temperature_2m":        file path
file        "2026-11-19T15:00"      45.0                     .../15/heat_spike_...
               │                       │                        │
BRONZE      payload.current.time    payload.current.         source_file
bronze.py   (text)                    temperature_2m         ingest_ts = 14:15
               │ CAST AS TIMESTAMP     │ CAST AS DOUBLE         │ current_timestamp()
SILVER      obs_ts = 15:00          temp_c = 45.0            ingest_ts = 14:15
silver.sql     │                       │                        │
          ┌────┴──────────┐       ┌────┴──────────┐             │
GOLD      ▼               ▼       ▼               ▼             ▼
gold.sql  obs_hour        obs_ts  avg_temp_c      temp_c ≥ 35   detected_at
          = 15:00         = 15:00 = 45.0          → 'HEAT'      = 14:15
          (hourly)       (alerts) (hourly)        (alerts)      (alerts)
             │                       │                             │
SERVE     map: latest hour        map color,                    alert: detected
          per city                MAX(temp_c) in email          in the last hour
```

In [0]:
# Did the row reach silver?
# show obs_ts, temp_c, ingest_ts, source_file from silver
# order by ingest_ts
display(spark.sql("""
    
"""))

In [0]:
# Did gold (weather_alerts) flag it? 
display(spark.sql("""
    
"""))

# 3. Act 2: bad records
Recall in Lecture 18, we set up 5 expectations on silver table. We will make up bad records that breaks the following rule
```sql
CONSTRAINT valid_city EXPECT (city IS NOT NULL) ON VIOLATION DROP ROW
```
Similar to how we made "heat_spike_wilmington.json", we will make 
- `folder/bad_city.json` such that its `["city"]` is None

use the same `folder` as defined in the first cell

Now: **Run now** on `weather-hourly`. Then open `weather-ETL` → latest update → `silver_observations` → **Expectations**: 1 dropped for `valid_city`.

# 4. After class: clean up

When the hourly schedule runs in that hour again, the real Wilmington reading lands in the same hour as the spike, and `gold_hourly_city` averages the two. To remove the demo data:

1. Run the cell below. It finds the demo files in any hour folder and only **lists** them; uncomment `os.remove` to delete them (permanent).
2. **Full refresh** the `weather-ETL` pipeline, so bronze and silver are rebuilt without them.
3. Unpause the hourly schedule.

In [0]:
LANDING = "/Volumes/dsc592/weather/landing/weather"
demo_files = []
for name in ["heat_spike_*.json", "bad_city.json"]:  
    demo_files += glob.glob(f"{LANDING}/*/*/*/*/{name}")

for p in demo_files:
    print(p)
    # os.remove(p)                                    # uncomment to delete